# Neon Heist Lab: NumPy + pandas + PyTorch

A playful but serious one-page project:
- simulate thousands of cyber-heists with vectorized NumPy
- build an ops dashboard with pandas
- train a PyTorch model to predict mission success
- run counterfactual edits to flip failed missions

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn

np.random.seed(7)
torch.manual_seed(7)

print('NumPy  :', np.__version__)
print('pandas :', pd.__version__)
print('PyTorch:', torch.__version__)

## 1) Vectorized Mission Simulator

In [ ]:
n = 8000
crew = np.random.randint(2, 9, n)
stealth = np.random.uniform(0.2, 1.0, n)
exploit_quality = np.random.beta(2.2, 1.6, n)
intel = np.random.uniform(0.1, 1.0, n)

security_tier = np.random.choice([1, 2, 3, 4], size=n, p=[0.25, 0.35, 0.25, 0.15])
target_value = np.random.lognormal(mean=3.7, sigma=0.55, size=n)

# Detection pressure rises with security tier and larger crews.
pressure = 0.65 * security_tier + 0.17 * crew + np.random.normal(0, 0.35, n)

# Mission score mixes linear + non-linear interactions.
score = (
    2.0 * stealth
    + 1.9 * exploit_quality
    + 1.3 * intel
    - 0.65 * pressure
    - 0.22 * np.maximum(crew - 5, 0)
    + 0.5 * np.sin(3.4 * stealth)
    + np.random.normal(0, 0.25, n)
)

p_success = 1.0 / (1.0 + np.exp(-score))
success = (np.random.rand(n) < p_success).astype(np.int64)

loot = target_value * (0.35 + 0.9 * exploit_quality) * success
heat = (1.2 - stealth + 0.28 * security_tier + 0.12 * crew + np.random.normal(0, 0.12, n))
heat = np.clip(heat, 0.0, None)

df = pd.DataFrame({
    'crew': crew,
    'stealth': stealth,
    'exploit_quality': exploit_quality,
    'intel': intel,
    'security_tier': security_tier,
    'target_value': target_value,
    'pressure': pressure,
    'p_success': p_success,
    'success': success,
    'loot': loot,
    'heat': heat,
})

df['risk_reward'] = df['loot'] / (1.0 + df['heat'])
df['op_class'] = pd.cut(df['risk_reward'], bins=[-1, 20, 60, 140, 1e9], labels=['cold', 'clean', 'bold', 'legend'])

print(df.head(8))
print('\nSuccess rate:', round(df['success'].mean() * 100, 2), '%')

## 2) pandas Ops Dashboard

In [ ]:
tier_report = (
    df.groupby('security_tier')
      .agg(
          missions=('success', 'size'),
          success_rate=('success', 'mean'),
          avg_loot=('loot', 'mean'),
          avg_heat=('heat', 'mean'),
          avg_rr=('risk_reward', 'mean')
      )
      .sort_values('avg_rr', ascending=False)
)
tier_report['success_rate'] = (tier_report['success_rate'] * 100).round(2)

crew_heatmap = pd.crosstab(df['crew'], df['security_tier'], values=df['success'], aggfunc='mean').round(3)

elite = df.nlargest(12, 'risk_reward')[['crew', 'stealth', 'exploit_quality', 'intel', 'security_tier', 'loot', 'heat', 'risk_reward', 'op_class']]

print('Tier report:')
print(tier_report)
print('\nSuccess matrix (crew x security_tier):')
print(crew_heatmap)
print('\nTop risk-reward missions:')
elite

## 3) PyTorch Mission Classifier

In [ ]:
features = ['crew', 'stealth', 'exploit_quality', 'intel', 'security_tier', 'pressure', 'target_value', 'heat']
X = df[features].to_numpy(dtype=np.float32)
y = df['success'].to_numpy(dtype=np.float32).reshape(-1, 1)

split = int(0.82 * len(df))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

mu = X_train.mean(axis=0, keepdims=True)
sigma = X_train.std(axis=0, keepdims=True) + 1e-6
X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 48),
    nn.GELU(),
    nn.Linear(48, 24),
    nn.GELU(),
    nn.Linear(24, 1),
)

opt = torch.optim.AdamW(model.parameters(), lr=0.008, weight_decay=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(1, 241):
    model.train()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 60 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test_t)
            test_probs = torch.sigmoid(test_logits)
            pred = (test_probs >= 0.5).float()
            acc = (pred.eq(y_test_t).float().mean()).item()
        print(f'Epoch {epoch:3d} | train_loss={loss.item():.4f} | test_acc={acc:.4f}')

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_test_t)).cpu().numpy().reshape(-1)

pred = (probs >= 0.5).astype(np.int64)
truth = y_test.reshape(-1).astype(np.int64)
accuracy = float((pred == truth).mean())
precision = float(((pred == 1) & (truth == 1)).sum() / max((pred == 1).sum(), 1))
recall = float(((pred == 1) & (truth == 1)).sum() / max((truth == 1).sum(), 1))

print('\nFinal metrics')
print('accuracy :', round(accuracy, 4))
print('precision:', round(precision, 4))
print('recall   :', round(recall, 4))

## 4) Counterfactual Generator: How To Flip A Failed Mission

In [ ]:
test_df = df.iloc[split:].copy().reset_index(drop=True)
test_df['pred_success_prob'] = probs
failed = test_df[test_df['pred_success_prob'] < 0.5].head(5).copy()

if failed.empty:
    print('No predicted failures in this run. Re-run to sample a different seed.')
else:
    Xf = failed[features].to_numpy(dtype=np.float32)
    Xf_norm = (Xf - mu) / sigma
    x = torch.tensor(Xf_norm, requires_grad=True)

    # Gradient ascent on success probability with a small movement budget.
    optimizer = torch.optim.Adam([x], lr=0.03)
    for _ in range(120):
        logits = model(x)
        prob = torch.sigmoid(logits)
        l2_penalty = 0.02 * ((x - torch.tensor(Xf_norm)) ** 2).mean()
        objective = -(prob.mean() - l2_penalty)
        optimizer.zero_grad()
        objective.backward()
        optimizer.step()

    x_new = x.detach().numpy() * sigma + mu
    with torch.no_grad():
        p_before = torch.sigmoid(model(torch.tensor(Xf_norm))).numpy().reshape(-1)
        p_after = torch.sigmoid(model(torch.tensor((x_new - mu) / sigma))).numpy().reshape(-1)

    out = failed[['crew', 'stealth', 'exploit_quality', 'intel', 'security_tier', 'heat', 'pred_success_prob']].copy()
    out['new_stealth'] = x_new[:, features.index('stealth')]
    out['new_exploit_quality'] = x_new[:, features.index('exploit_quality')]
    out['new_intel'] = x_new[:, features.index('intel')]
    out['p_before'] = p_before
    out['p_after'] = p_after
    out['delta'] = out['p_after'] - out['p_before']
    out.sort_values('delta', ascending=False).reset_index(drop=True)